# Carbon Credit Compliance RAG Assistant

A retrieval-augmented generation (RAG) system that answers questions about India's
CCTS carbon-credit rules for waste projects — grounded in an official BEE/CCTS
knowledge document, with source citations.

This notebook keeps the **full build journey**, including the failures and the fixes,
because that is where the real understanding lives.

## The mental map — remember these five words

**Indexing (done once):**  `Chunk → Embed → Store`
**Query (every question):**  `Retrieve → Generate`

| Stage | What it does | Intuition |
|---|---|---|
| **Chunk** | Split the document into small passages | Like index cards — one topic per card |
| **Embed** | Turn each chunk into a number vector | Similar *meaning* lands nearby, not similar words |
| **Store** | Put vectors in a searchable index (FAISS) | A smart index that finds nearest neighbours fast |
| **Retrieve** | Embed the question, find nearest chunks | Pull the cards most related to the question |
| **Generate** | LLM answers using ONLY the retrieved text | Grounded in the document, not the model's memory |

## The five insights this notebook demonstrates
1. **Retrieval is imperfect** — top-3 + the LLM together are robust; no single chunk is.
2. **Chunking strategy decides retrieval quality** — word-count vs structure-aware.
3. **The embedding model matters** — MiniLM vs mpnet.
4. **Query phrasing matters** — short abstract queries ("what is X?") are the hardest.
5. **Grounding actually works** — verified with a hallucination test (it refuses
   questions whose answer is not in the document).

---
## Setup
Install the stack. `sentence-transformers` makes embeddings locally (no API);
`faiss-cpu` is the vector search library.

In [3]:
# The RAG stack - all lightweight, runs on Colab free tier
!pip install -q sentence-transformers faiss-cpu

# sentence-transformers -> turns text into embeddings (local, free, no API)
# faiss-cpu -> Facebook's vector search library (fast similarity search)

## Stage 1 — Chunk

Why not feed the whole document to the LLM? Two reasons: it can be too long, and a
small question ("what is additionality?") only needs one paragraph, not the whole file.

**Overlap** is deliberate: each chunk shares a few words with the previous one, so a
sentence split across a boundary is not lost.

**Chunk size is a trade-off** — small chunks give precise retrieval but little context;
large chunks give context but more noise. We start at 150 words and revisit this later.

In [4]:
# Load the document
with open('ccts_waste_methodology.md', 'r') as f:
    document = f.read()

print(f'Document length: {len(document)} characters')
print(f'Roughly {len(document.split())} words')

# Simple chunking: split into overlapping windows of words
def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into overlapping chunks.

    chunk_size: how many words per chunk
    overlap: how many words each chunk shares with the previous one,
             so a sentence split across a boundary is not lost
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # step forward, minus the overlap
    return chunks

chunks = chunk_text(document)
print(f'\nCreated {len(chunks)} chunks')
print(f'\n--- First chunk ---\n{chunks[0][:300]}...')
print(f'\n--- Second chunk (note the overlap with first) ---\n{chunks[1][:300]}...')

Document length: 7033 characters
Roughly 1055 words

Created 9 chunks

--- First chunk ---
# Carbon Credit Trading Scheme (CCTS) — Waste Handling Offset Methodology ## Reference knowledge base for compliance guidance > Source basis: Bureau of Energy Efficiency (BEE), Detailed Procedure for the > Offset Mechanism under CCTS (Version 1, March 2025); Ministry of Power / > MoEFCC notification...

--- Second chunk (note the overlap with first) ---
the Compliance Mechanism (mandatory, for large industrial emitters) and the Offset Mechanism (voluntary, for everyone else). The Offset Mechanism lets "non-obligated entities" — organisations not covered under the compliance mechanism, such as waste operators, societies, and small factories — regist...


## Stage 2 — Embed

Each chunk becomes a list of 384 numbers — its "meaning fingerprint". Two chunks with
similar meaning get similar fingerprints, **even if they use different words**. This is
what makes RAG better than keyword search: it matches *meaning*, not spelling.

In [5]:
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model (runs locally, no API needed)
# This model turns any text into a 384-number vector
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded')

# Embed all our chunks
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)

print(f'\nShape: {chunk_embeddings.shape}')
# (9, 384) -> 9 chunks, each turned into 384 numbers
print(f'\nFirst chunk, first 10 numbers of its embedding:')
print(chunk_embeddings[0][:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Shape: (9, 384)

First chunk, first 10 numbers of its embedding:
[-0.05626558  0.02658429 -0.0285578   0.01645369  0.07055455  0.02927887
  0.03150703  0.05194128 -0.01179361 -0.01153248]


### Proof that embeddings capture meaning

Two questions that mean the same thing but share almost no words should score HIGH;
an unrelated sentence should score LOW.

**Result observed:** similar-meaning pair ≈ **0.64**, unrelated pair ≈ **0.04**.
That gap is the whole foundation of RAG — remember it: *similar meaning → high score → nearby.*

In [6]:
from sentence_transformers import util

# Two questions that MEAN similar things but use different words
q1 = "Can a housing society earn carbon credits for composting?"
q2 = "Is a residential complex eligible for CCCs from organic waste?"
# A totally unrelated sentence
q3 = "What is the weather in Mumbai today?"

emb = embedder.encode([q1, q2, q3])

# Cosine similarity: 1.0 = identical meaning, 0 = unrelated
print('q1 vs q2 (similar meaning):', util.cos_sim(emb[0], emb[1]).item())
print('q1 vs q3 (unrelated):     ', util.cos_sim(emb[0], emb[2]).item())

q1 vs q2 (similar meaning): 0.6385462284088135
q1 vs q3 (unrelated):      0.04431416839361191


## Stage 3 — Store (FAISS)

Now we put all chunk vectors into a FAISS index. With 9 chunks we could compare one by
one, but FAISS scales to thousands — think of it as a smart index that jumps straight to
the nearest matches. `IndexFlatL2` does exact nearest-neighbour search by distance
(lower distance = more similar).

In [7]:
import faiss
import numpy as np

# FAISS needs float32 vectors
embeddings_np = np.array(chunk_embeddings).astype('float32')

# Build the index. IndexFlatL2 = exact nearest-neighbour search by distance.
# 384 = the dimension of our embeddings (must match)
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

print(f'Index built with {index.ntotal} chunks')
print(f'Each chunk is a {dimension}-dimensional vector')

Index built with 9 chunks
Each chunk is a 384-dimensional vector


## Stage 4 — Retrieve (tested WITHOUT the LLM first)

**This is the step most people skip — and it's why their RAG silently breaks.**

If we add the LLM now and an answer is wrong, we can't tell whether *retrieval* was wrong
(wrong chunk) or the *LLM* was wrong (right chunk, bad answer). So we test retrieval alone.

Below, `retrieve` embeds the question into the same 384-dim space and returns the nearest
chunks with their distances.

In [8]:
def retrieve(question, k=3):
    """Find the k most relevant chunks for a question.

    Steps:
    1. Embed the question (same model as the chunks)
    2. Search the index for the nearest chunk vectors
    3. Return those chunks
    """
    # Embed the question into the same 384-dim space
    q_emb = embedder.encode([question]).astype('float32')

    # Search: returns distances and the indices of nearest chunks
    distances, indices = index.search(q_emb, k)

    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        results.append({
            'rank': rank + 1,
            'distance': float(dist),   # lower = more similar
            'chunk': chunks[idx]
        })
    return results

# Test it with a real question
question = "Can a housing society earn carbon credits for composting?"
results = retrieve(question)

print(f'Question: {question}\n')
for r in results:
    print(f"--- Rank {r['rank']} (distance {r['distance']:.2f}) ---")
    print(r['chunk'][:250], '...\n')

Question: Can a housing society earn carbon credits for composting?

--- Rank 1 (distance 0.61) ---
to composting or biogas avoids most of those methane emissions. Key quantities a waste project must track: - Tonnes of each waste type diverted from landfill. - The destination and treatment method (compost, biogas, recycling). - The fraction of the  ...

--- Rank 2 (distance 0.71) ---
earn carbon credits for its waste?** Potentially, if it runs a qualifying waste-diversion project (for example on-site composting) that meets the eligibility rules — started on or after 1 January 2025, additional, not double-counted, and following an ...

--- Rank 3 (distance 0.79) ---
the Compliance Mechanism (mandatory, for large industrial emitters) and the Offset Mechanism (voluntary, for everyone else). The Offset Mechanism lets "non-obligated entities" — organisations not covered under the compliance mechanism, such as waste  ...



### Iteration 1 — retrieval is imperfect (the first failure)

Stress-testing with `k=1` (hardest setting) on three questions:

- **"When must the project have started?"** → perfect, exact answer on top.
- **"What is additionality?"** → MISS (brought a composting chunk, not the definition).
- **"Who verifies the reductions?"** → MISS (brought a background chunk).

**Lesson:** short, abstract questions are the hardest, and `k=1` is unforgiving. This is
not a bug — it's the real behaviour of retrieval, and the reason we retrieve more than one.

In [9]:
# Try a few different questions to stress-test retrieval
for q in [
    "What is additionality?",
    "Who verifies the carbon reductions?",
    "When must the project have started?",
]:
    top = retrieve(q, k=1)[0]
    print(f"Q: {q}")
    print(f"   -> {top['chunk'][:150]}...\n")

Q: What is additionality?
   -> earn carbon credits for its waste?** Potentially, if it runs a qualifying waste-diversion project (for example on-site composting) that meets the elig...

Q: Who verifies the carbon reductions?
   -> the Compliance Mechanism (mandatory, for large industrial emitters) and the Offset Mechanism (voluntary, for everyone else). The Offset Mechanism lets...

Q: When must the project have started?
   -> **Start date:** The project activity must have commenced on or after 1 January 2025. - **Additionality:** The emission reductions must be additional —...



### Iteration 2 — why we retrieve top-3, not top-1

The same failing questions, but with `k=3`.

- **"Who verifies?"** → the correct chunk now appears at rank 2. Fixed by k=3.
- **"What is additionality?"** → still not in the top-3. A genuinely stubborn case.

**Lesson (core RAG principle):** retrieve a few chunks and let the LLM choose — don't bet
on a single chunk being perfect.

In [10]:
# The same failing questions, but retrieve top 3 instead of top 1
for q in ["What is additionality?", "Who verifies the carbon reductions?"]:
    print(f"Q: {q}")
    results = retrieve(q, k=3)
    for r in results:
        print(f"   Rank {r['rank']} (dist {r['distance']:.2f}): {r['chunk'][:110]}...")
    print()

Q: What is additionality?
   Rank 1 (dist 1.59): earn carbon credits for its waste?** Potentially, if it runs a qualifying waste-diversion project (for example...
   Rank 2 (dist 1.81): project cycle (the steps to earn credits) A project passes through these stages before CCCs are issued: 1. **R...
   Rank 3 (dist 1.82): # Carbon Credit Trading Scheme (CCTS) — Waste Handling Offset Methodology ## Reference knowledge base for comp...

Q: Who verifies the carbon reductions?
   Rank 1 (dist 0.96): the Compliance Mechanism (mandatory, for large industrial emitters) and the Offset Mechanism (voluntary, for e...
   Rank 2 (dist 1.01): **Start date:** The project activity must have commenced on or after 1 January 2025. - **Additionality:** The ...
   Rank 3 (dist 1.01): earn carbon credits for its waste?** Potentially, if it runs a qualifying waste-diversion project (for example...



### Enhancement 1 — structure-aware chunking

Our first chunker split blindly by word count, so a one-line definition got averaged
together with unrelated rules inside a big chunk. Fix: split on the document's **natural
structure** (blank lines / headings) so each concept keeps its own chunk.

This helped several questions — but, as we'll see, additionality is still stubborn.

In [11]:
# Smarter chunking: split on structure (blank lines / headings),
# so each concept stays in its own chunk instead of being averaged
# together with unrelated rules.

def chunk_by_structure(text, min_words=20, max_words=200):
    """Split on double-newlines (paragraphs/sections), then merge
    tiny pieces and split oversized ones."""
    # Split on blank lines - these separate paragraphs and list blocks
    raw_blocks = [b.strip() for b in text.split('\n\n') if b.strip()]

    chunks, current = [], ''
    for block in raw_blocks:
        combined = (current + '\n\n' + block).strip()
        if len(combined.split()) <= max_words:
            current = combined
        else:
            if current:
                chunks.append(current)
            current = block
    if current:
        chunks.append(current)

    # Drop chunks that are too tiny to be meaningful on their own
    chunks = [c for c in chunks if len(c.split()) >= min_words]
    return chunks

chunks_v2 = chunk_by_structure(document)
print(f'Structure-based chunks: {len(chunks_v2)}')

# Re-embed and rebuild the index with the new chunks
emb_v2 = embedder.encode(chunks_v2, show_progress_bar=True).astype('float32')
index_v2 = faiss.IndexFlatL2(emb_v2.shape[1])
index_v2.add(emb_v2)

# Retrieve helper for the new index
def retrieve_v2(question, k=3):
    q_emb = embedder.encode([question]).astype('float32')
    distances, indices = index_v2.search(q_emb, k)
    return [{'rank': i+1, 'distance': float(d), 'chunk': chunks_v2[idx]}
            for i, (idx, d) in enumerate(zip(indices[0], distances[0]))]

# Test the previously-failing question
print('\nQ: What is additionality?\n')
for r in retrieve_v2("What is additionality?"):
    print(f"Rank {r['rank']} (dist {r['distance']:.2f}): {r['chunk'][:130]}...\n")

Structure-based chunks: 6


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What is additionality?

Rank 1 (dist 1.68): - Tonnes of each waste type diverted from landfill.
- The destination and treatment method (compost, biogas, recycling).
- The fra...

Rank 2 (dist 1.81): 1. **Registration:** The entity registers on the Indian Carbon Market (ICM)
   portal as a non-obligated entity.
2. **Project Desi...

Rank 3 (dist 1.84): # Carbon Credit Trading Scheme (CCTS) — Waste Handling Offset Methodology
## Reference knowledge base for compliance guidance

> S...



### Enhancement 2 — a stronger embedding model

`all-MiniLM-L6-v2` is small and fast but weaker. `all-mpnet-base-v2` is larger and much
better at nuanced meaning. We re-embed the structure-based chunks with it.

**Lesson:** the two biggest levers in RAG quality are **chunking** and the **embedding
model**. We've now tried both.

In [12]:
# A stronger embedding model - better at capturing nuanced meaning
# Slightly slower, but much better retrieval quality
embedder_v2 = SentenceTransformer('all-mpnet-base-v2')
print('Stronger embedder loaded')

# Re-embed the structure-based chunks with the better model
emb_v3 = embedder_v2.encode(chunks_v2, show_progress_bar=True).astype('float32')
index_v3 = faiss.IndexFlatL2(emb_v3.shape[1])
index_v3.add(emb_v3)

def retrieve_v3(question, k=3):
    q_emb = embedder_v2.encode([question]).astype('float32')
    distances, indices = index_v3.search(q_emb, k)
    return [{'rank': i+1, 'distance': float(d), 'chunk': chunks_v2[idx]}
            for i, (idx, d) in enumerate(zip(indices[0], distances[0]))]

# Test the stubborn question
print('\nQ: What is additionality?\n')
for r in retrieve_v3("What is additionality?"):
    print(f"Rank {r['rank']} (dist {r['distance']:.2f}): {r['chunk'][:130]}...\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stronger embedder loaded


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What is additionality?

Rank 1 (dist 1.56): - Tonnes of each waste type diverted from landfill.
- The destination and treatment method (compost, biogas, recycling).
- The fra...

Rank 2 (dist 1.66): - **Baseline:** What the emissions would have been without the project. For
  waste, the baseline is usually the methane that woul...

Rank 3 (dist 1.71): # Carbon Credit Trading Scheme (CCTS) — Waste Handling Offset Methodology
## Reference knowledge base for compliance guidance

> S...



### Diagnosis — WHY additionality keeps failing

Before tuning blindly, look at where the answer actually lives. Searching the chunks for
the word "additional" reveals the root cause:

The clean definition of additionality **does not sit in any single chunk**. It appears as
one bullet inside a big eligibility block (mixed with start-date, double-counting, etc.),
so the chunk's *average meaning* is "a mix of eligibility rules", not "additionality".

**This is the deepest RAG lesson here:** *RAG is only as good as its chunking. If the answer
isn't in one clean chunk, no embedding model or query trick fully rescues it.*

In [13]:
# Find which chunk(s) actually contain the additionality definition
for i, c in enumerate(chunks_v2):
    if 'additional' in c.lower():
        print(f"--- Chunk {i} contains 'additional' ---")
        print(c[:400])
        print()

--- Chunk 1 contains 'additional' ---
Waste handling and disposal is one of the sectors covered under the Offset
Mechanism. Projects that divert waste from landfill, capture landfill gas, or
process organic waste into compost or biogas may qualify.

---

## 2. Core eligibility rules

A project must satisfy all of the following to be eligible:

- **Start date:** The project activity must have commenced on or after
  1 January 2025.
- *

--- Chunk 4 contains 'additional' ---
- Tonnes of each waste type diverted from landfill.
- The destination and treatment method (compost, biogas, recycling).
- The fraction of the waste that is organic / degradable.
- Any emissions from transport and processing (these count as project emissions
  and reduce the net credit).

---

## 6. Compliance timeline (context)

- March 2025: BEE released the Offset Mechanism Detailed Procedure (



### Confirming it's the query phrasing (the semantic gap)

The question "What is additionality?" is 3 abstract words; the answer is a full sentence
with different vocabulary ("would not have happened without carbon finance"). That
mismatch is called the **semantic gap**.

Testing fuller, intent-based phrasings shows retrieval improves as the query gets richer —
proof that *how you phrase the question* changes what comes back.

In [14]:
# Test whether the problem is just the ultra-short abstract phrasing
questions = [
    "What is additionality?",                          # abstract, short
    "What does additional mean for a carbon project?",  # fuller
    "Does a project qualify if it would have happened anyway?",  # intent-based
]
for q in questions:
    print(f"Q: {q}")
    top = retrieve_v3(q, k=1)[0]
    has_def = 'additional' in top['chunk'].lower()
    print(f"   Top chunk has 'additional': {has_def}")
    print(f"   -> {top['chunk'][:120]}...\n")

Q: What is additionality?
   Top chunk has 'additional': True
   -> - Tonnes of each waste type diverted from landfill.
- The destination and treatment method (compost, biogas, recycling)....

Q: What does additional mean for a carbon project?
   Top chunk has 'additional': True
   -> - Tonnes of each waste type diverted from landfill.
- The destination and treatment method (compost, biogas, recycling)....

Q: Does a project qualify if it would have happened anyway?
   Top chunk has 'additional': True
   -> Waste handling and disposal is one of the sectors covered under the Offset
Mechanism. Projects that divert waste from la...



## Stage 5 — Generate (add the LLM)

Now RAG becomes complete. We load a small local instruction-tuned LLM (Qwen2.5-1.5B),
the same family we used elsewhere, small enough for a free T4 GPU.

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Small instruction-tuned LLM - fits easily on a T4
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto")
print('LLM loaded')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded


### The heart of RAG — the augmented prompt

The key move: we hand the LLM the retrieved chunks and instruct it to answer using **ONLY**
that context, and to say "I don't have that information" otherwise. This is the "Augmented"
in RAG — the model answers from *retrieved text*, not its own memory.

**The payoff:** even though retrieval never put additionality's definition on top, the LLM
reads the top-3 chunks, finds the relevant line, and answers correctly. This is the whole
point — *retrieval + LLM together are robust; neither is perfect alone.*

In [17]:
def rag_answer(question, k=3):
    # 1. RETRIEVE - get the most relevant chunks
    results = retrieve_v3(question, k=k)
    context = "\n\n---\n\n".join([r['chunk'] for r in results])

    # 2. AUGMENT - build a prompt that forces the LLM to use ONLY the context
    prompt = f"""You are a carbon credit compliance assistant. Answer the question using ONLY the context below. If the answer is not in the context, say "I don't have that information in the provided documents." Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""

    # 3. GENERATE - let the LLM write the grounded answer
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=256, temperature=0.1)
    response = tokenizer.decode(
        out[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return response, results

# Test it on the question that retrieval struggled with
answer, sources = rag_answer("What is additionality?")
print("ANSWER:\n", answer)
print("\n--- Sources used (top 3 chunks) ---")
for r in sources:
    print(f"[Rank {r['rank']}] {r['chunk'][:80]}...")

ANSWER:
 A project is considered additional if it needs the carbon finance to occur. If a factory was already planning to compost its waste to save money, the reductions from this action are not additional and therefore do not qualify for carbon credits under the scheme.

--- Sources used (top 3 chunks) ---
[Rank 1] - Tonnes of each waste type diverted from landfill.
- The destination and treatm...
[Rank 2] - **Baseline:** What the emissions would have been without the project. For
  wa...
[Rank 3] # Carbon Credit Trading Scheme (CCTS) — Waste Handling Offset Methodology
## Ref...


### The critical test — is it really grounded? (hallucination test)

The most important test in the whole notebook. We ask things whose answer is **not** in the
document (a carbon-credit price, how to register a company). A properly grounded RAG must
refuse; a leaky one would answer from the model's training memory.

**Result:** both out-of-scope questions returned *"I don't have that information in the
provided documents"*, while the in-document control (additionality) answered correctly.
This proves the answers come from the document, not the model — exactly what a compliance
tool needs (a wrong price could be a legal problem).

In [18]:
# THE critical test: ask something NOT in the document.
# A properly grounded RAG must refuse. A leaky one will hallucinate.

test_questions = [
    "What is the current price of one carbon credit in rupees?",   # not in doc
    "How do I register a company in India?",                       # unrelated
    "What is additionality?",                                      # in doc (control)
]

for q in test_questions:
    answer, _ = rag_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n{'-'*60}")

Q: What is the current price of one carbon credit in rupees?
A: I don't have that information in the provided documents.
------------------------------------------------------------
Q: How do I register a company in India?
A: I don't have that information in the provided documents.
------------------------------------------------------------
Q: What is additionality?
A: A project is considered additional if it needs the carbon finance to occur. If a factory was already planning to compost its waste to save money, the reductions from this action are not additional and therefore do not qualify for carbon credits under the scheme.
------------------------------------------------------------


## Enhancement 3 — source citations

For a compliance tool, "where did this rule come from?" matters as much as the answer.
Here we tag each chunk with the section heading it falls under, so every answer can cite
its sources. A nice side effect: adding section context also nudged retrieval quality up.

In [19]:
import re

def chunk_with_sections(text, max_words=200, min_words=20):
    """Split by structure, but tag each chunk with the section heading
    it falls under - so we can cite the source later."""
    raw_blocks = [b.strip() for b in text.split('\n\n') if b.strip()]

    chunks_meta = []
    current_section = "Introduction"
    current = ''

    for block in raw_blocks:
        # Detect a markdown heading (## 2. Core eligibility rules)
        heading = re.match(r'^#+\s+(.*)', block)
        if heading:
            current_section = heading.group(1).strip()

        combined = (current + '\n\n' + block).strip()
        if len(combined.split()) <= max_words:
            current = combined
        else:
            if current and len(current.split()) >= min_words:
                chunks_meta.append({'text': current, 'section': current_section})
            current = block
    if current and len(current.split()) >= min_words:
        chunks_meta.append({'text': current, 'section': current_section})

    return chunks_meta

chunks_meta = chunk_with_sections(document)
print(f'{len(chunks_meta)} chunks with section labels\n')
for i, c in enumerate(chunks_meta):
    print(f"[{i}] Section: {c['section']!r}  ({len(c['text'].split())} words)")

6 chunks with section labels

[0] Section: '1. What the CCTS Offset Mechanism is'  (189 words)
[1] Section: '3. The project cycle (the steps to earn credits)'  (188 words)
[2] Section: '4. Baseline and monitoring — the two halves of a methodology'  (199 words)
[3] Section: '5. Waste handling — how reductions arise'  (161 words)
[4] Section: '7. Common questions'  (200 words)
[5] Section: '7. Common questions'  (118 words)


### Rebuild the index with section metadata

Re-embed the section-tagged chunks (using the stronger mpnet model) and rebuild the FAISS
index, keeping each chunk's section label alongside for citation.

In [20]:
# Re-embed using the text, but keep the section metadata alongside
chunk_texts = [c['text'] for c in chunks_meta]
emb_final = embedder_v2.encode(chunk_texts, show_progress_bar=True).astype('float32')

index_final = faiss.IndexFlatL2(emb_final.shape[1])
index_final.add(emb_final)

def retrieve_cited(question, k=3):
    """Retrieve chunks WITH their section labels for citation."""
    q_emb = embedder_v2.encode([question]).astype('float32')
    distances, indices = index_final.search(q_emb, k)
    return [{'section': chunks_meta[idx]['section'],
             'text': chunks_meta[idx]['text'],
             'distance': float(d)}
            for idx, d in zip(indices[0], distances[0])]

# Quick check
for r in retrieve_cited("Who verifies the reductions?"):
    print(f"[{r['section']}] {r['text'][:70]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[7. Common questions] **Who verifies the reductions?**
An Accredited Carbon Verification Age...
[4. Baseline and monitoring — the two halves of a methodology] 1. **Registration:** The entity registers on the Indian Carbon Market ...
[5. Waste handling — how reductions arise] - **Baseline:** What the emissions would have been without the project...


### Answer with citations

The final RAG function: retrieve section-tagged chunks, generate a grounded answer, and
return the unique source sections it drew from. Now the assistant shows its work.

In [21]:
def rag_answer_cited(question, k=3):
    results = retrieve_cited(question, k=k)
    context = "\n\n---\n\n".join(
        [f"[Source: {r['section']}]\n{r['text']}" for r in results])

    prompt = f"""You are a carbon credit compliance assistant. Answer using ONLY the context below. If the answer is not in the context, say "I don't have that information in the provided documents." Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=256, temperature=0.1)
    answer = tokenizer.decode(
        out[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

    # Show which sections the answer drew from
    sources = list(dict.fromkeys([r['section'] for r in results]))  # unique, ordered
    return answer, sources

answer, sources = rag_answer_cited("What is additionality?")
print("ANSWER:\n", answer)
print("\nSOURCES:")
for s in sources:
    print(f"  - {s}")

ANSWER:
 A project is additional if it needed the carbon finance to happen. If a factory was already going to compost its waste to save money, the reductions are not additional and do not qualify.

SOURCES:
  - 7. Common questions
  - 5. Waste handling — how reductions arise
  - 1. What the CCTS Offset Mechanism is


## The demo UI (Gradio)

A simple interface: type a question, get a grounded answer plus the document sections it
came from. `share=True` gives a public link for demos.

Try the last example ("price of a carbon credit") live — it should refuse, which is the
most convincing thing to show: the system knows the limits of its own knowledge.

In [22]:
!pip install -q gradio

import gradio as gr

def rag_ui(question):
    """Wrapper for the Gradio interface - returns answer + formatted sources."""
    if not question.strip():
        return "Please enter a question.", ""

    answer, sources = rag_answer_cited(question, k=3)

    # Format sources as a clean list
    sources_text = "\n".join([f"• {s}" for s in sources])
    return answer, sources_text

# Build the interface
with gr.Blocks(title="Carbon Credit Compliance Assistant") as demo:
    gr.Markdown(
        """
        # 🌱 Carbon Credit Compliance Assistant
        Ask about India's CCTS Offset Mechanism for waste projects.
        Answers are grounded in official BEE/CCTS documents — with sources cited.

        *Prototype for guidance only. Final eligibility is decided by BEE and an accredited agency (ACVA).*
        """
    )

    with gr.Row():
        question = gr.Textbox(
            label="Your question",
            placeholder="e.g. Can a housing society earn carbon credits for composting?",
            lines=2
        )

    ask_btn = gr.Button("Ask", variant="primary")

    with gr.Row():
        answer_out = gr.Textbox(label="Answer", lines=6)
    with gr.Row():
        sources_out = gr.Textbox(label="Sources (document sections)", lines=3)

    # Example questions users can click
    gr.Examples(
        examples=[
            "What is additionality?",
            "When must the project have started?",
            "Who verifies the carbon reductions?",
            "Can a housing society earn carbon credits for composting?",
            "What is the current price of a carbon credit?",  # not in doc - tests grounding
        ],
        inputs=question
    )

    ask_btn.click(fn=rag_ui, inputs=question, outputs=[answer_out, sources_out])
    question.submit(fn=rag_ui, inputs=question, outputs=[answer_out, sources_out])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8048f0a7cbf0036fe9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Summary — what this build taught

| Stage | Enhancement / failure | Fix |
|---|---|---|
| Chunk | Blind word-count mixed concepts | Structure-aware chunking |
| Embed | MiniLM weak on nuance | Switched to mpnet |
| Retrieve | k=1 too strict; abstract queries miss | Retrieve top-3 |
| Retrieve | Additionality not in any clean chunk | Accepted; LLM recovers it from top-3 |
| Generate | Risk of using model memory | Grounded prompt + hallucination test |
| Cite | No provenance | Section-tagged chunks + source list |

**The one-line takeaway:** a RAG system is only as good as its retrieval, and retrieval is
only as good as its chunking — but a grounded LLM over the top-k makes the whole thing
robust even when retrieval is imperfect.

*Prototype for guidance only. Final carbon-credit eligibility is decided by BEE and an
accredited verification agency (ACVA).*